# Let's now start analyzing by state to better see a trend. Here are the states we will focus on as per ChatGPT

Here is a stronger list limited to events occurring within 1995–2017:

### Suggested States for Analysis

**California** is the strongest overall choice. The state experienced major pertussis epidemics in 2010 and 2014, followed by the passage of Senate Bill 277 in 2015, which eliminated personal-belief exemptions for required school vaccinations. This supports both outbreak and policy analysis. See [California SB 277](https://leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=201520160SB277).

**Minnesota** experienced a major measles outbreak in 2017 associated with reduced MMR vaccination in a localized community. Because the event occurred during the final year of your coverage and CDC measles datasets, Minnesota provides one of the clearest comparisons in your study. See the [CDC outbreak report](https://stacks.cdc.gov/view/cdc/47978).

**Ohio** experienced a large measles outbreak in an underimmunized Amish community in 2014. It is useful for showing how localized vaccination gaps can produce substantial outbreaks even when statewide coverage appears high.

**Washington** declared a pertussis epidemic in 2012 after reported cases increased sharply. This event falls directly within the years covered by both your pertussis and vaccination datasets, making Washington one of the best options for a state-level time-series analysis. See the [Washington Department of Health](https://doh.wa.gov/you-and-your-family/illness-and-disease-z/pertussis-whooping-cough/pertussis-notifiable-condition).

**New York** experienced a large mumps outbreak beginning in 2009, particularly within close-knit communities in New York City and surrounding areas. It provides a useful example of an outbreak occurring despite relatively high vaccination coverage.

**Arkansas** experienced a large mumps outbreak during 2016–2017. Because these years are present in your mumps case series, Arkansas is especially useful for examining whether statewide coverage measures concealed vulnerable communities.

For the cleanest analysis using the data you already have, I would prioritize:

1. **California — pertussis, 2010 and 2014**
2. **Washington — pertussis, 2012**
3. **Minnesota — measles, 2017**
4. **Arkansas — mumps, 2016–2017**

Ohio and New York are historically valuable, but their major events fall inside gaps in your current Tycho measles or mumps series, so they would require supplemental CDC case data.
                                                                                      |


In [2]:
import pandas as pd

cases_df = pd.read_csv('../app/data/tycho_cases.csv')
cases_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases
0,1995,AK,NaN,13.0,1.0
1,1995,AL,NaN,4.0,38.0
2,1995,AR,2.0,10.0,41.0
3,1995,AZ,10.0,2.0,151.0
4,1995,CA,108.0,206.0,463.0


In [3]:
coverage_df = pd.read_csv('../app/data/nis_vacc_coverage.csv')
coverage_df.head()

,state,year,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,AK,1995,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,AK,1996,84.55,260.0,225.0,84.55,260.0,225.0,78.25,260.0,210.0
2,AK,1997,87.41,291.0,257.0,87.41,291.0,257.0,80.98,291.0,242.0
3,AK,1998,87.06,34.0,30.0,87.06,34.0,30.0,82.01,34.0,28.0
4,AK,1999,90.67,349.0,321.0,90.67,349.0,321.0,83.54,349.0,296.0


In [5]:
yearly_coverage_cases_df = pd.merge(cases_df, coverage_df, on=['year', 'state'])
yearly_coverage_cases_df.head()

,year,state,measles_cases,mumps_cases,pertussis_cases,measles_coverage_pct,measles_n,measles_n_vaccinated,mumps_coverage_pct,mumps_n,mumps_n_vaccinated,pertussis_coverage_pct,pertussis_n,pertussis_n_vaccinated
0,1995,AK,NaN,13.0,1.0,89.86,194.0,177.0,89.86,194.0,177.0,77.58,194.0,159.0
1,1995,AL,NaN,4.0,38.0,88.77,419.0,386.0,88.77,419.0,386.0,78.93,419.0,357.0
2,1995,AR,2.0,10.0,41.0,90.43,237.0,218.0,90.43,237.0,218.0,77.24,237.0,190.0
3,1995,AZ,10.0,2.0,151.0,82.29,372.0,319.0,82.29,372.0,319.0,74.52,372.0,296.0
4,1995,CA,108.0,206.0,463.0,90.32,681.0,627.0,90.32,681.0,627.0,76.76,681.0,542.0


In [21]:
california_yearly_coverage_cases_df = yearly_coverage_cases_df[yearly_coverage_cases_df['state'] == 'CA']

In [22]:
washington_yearly_coverage_cases_df = yearly_coverage_cases_df[yearly_coverage_cases_df['state'] == 'WA']

In [23]:
minnesota_yearly_coverage_cases_df = yearly_coverage_cases_df[yearly_coverage_cases_df['state'] == 'MN']

In [24]:
arkansas_yearly_coverage_cases_df = yearly_coverage_cases_df[yearly_coverage_cases_df['state'] == 'AR']

In [25]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go


def graph_coverage_cases_df(df, state):

    diseases = ["measles", "mumps", "pertussis"]

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        specs=[
            [{"secondary_y": True}],
            [{"secondary_y": True}],
            [{"secondary_y": True}]
        ],
        subplot_titles=["Measles", "Mumps", "Pertussis"],
        vertical_spacing=0.10
    )

    colors = {
        "measles": "#E74C3C",
        "mumps": "#3498DB",
        "pertussis": "#2ECC71"
    }

    for row, disease in enumerate(diseases, start=1):

        # Annual cases
        fig.add_trace(
            go.Bar(
                x=df["year"],
                y=df[f"{disease}_cases"],
                name=f"{disease.title()} cases",
                marker_color=colors[disease],
                opacity=0.65,
                hovertemplate=(
                    "Year: %{x}<br>"
                    "Cases: %{y:,.0f}"
                    "<extra></extra>"
                )
            ),
            row=row,
            col=1,
            secondary_y=False
        )

        # Annual vaccination coverage
        fig.add_trace(
            go.Scatter(
                x=df["year"],
                y=df[f"{disease}_coverage_pct"],
                name=f"{disease.title()} coverage",
                mode="lines+markers",
                line=dict(color="black", width=2),
                marker=dict(size=6),
                hovertemplate=(
                    "Year: %{x}<br>"
                    "Coverage: %{y:.2f}%"
                    "<extra></extra>"
                )
            ),
            row=row,
            col=1,
            secondary_y=True
        )

        fig.update_yaxes(
            title_text="Cases",
            row=row,
            col=1,
            secondary_y=False
        )

        fig.update_yaxes(
            title_text="Coverage (%)",
            row=row,
            col=1,
            secondary_y=True
        )

    fig.update_xaxes(
        title_text="Year",
        row=3,
        col=1,
        dtick=1
    )

    fig.update_layout(
        title=f"Annual Vaccination Coverage and Reported Cases In {state}",
        template="plotly_white",
        height=900,
        hovermode="x unified",
        legend_title="Measure",
        bargap=0.20
    )

    fig.show()

In [26]:
graph_coverage_cases_df(california_yearly_coverage_cases_df, "California")

In [27]:
graph_coverage_cases_df(washington_yearly_coverage_cases_df, 'Washington')

In [28]:
graph_coverage_cases_df(minnesota_yearly_coverage_cases_df, 'Minnesota')

In [29]:
graph_coverage_cases_df(arkansas_yearly_coverage_cases_df, 'Arkansas')